# Week 5 — ATNF Pulsar Catalog (HEASARC CSV)

This notebook loads **`atnf_pulsar_heasarc.csv`** (HEASARC ATNF / `atnfpulsar` dump) and walks through the usual first-look questions with **pandas**.

**Run the notebook from the `Week 5` folder** so the relative path to the CSV works.

## Setup

**What question does this answer?**  
Do we have the libraries we need, and a working pandas import?

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
print("pandas:", pd.__version__)

**What does your dataset look like? `head()`, `info()`**  
We load the CSV into a DataFrame called `pulsars`, print the first rows, then ask pandas for column dtypes and how many non-null values each column has.

In [ ]:
pulsars = pd.read_csv("atnf_pulsar_heasarc.csv")
print(pulsars.head())
print()
pulsars.info()

**What's the distribution of your most important column?**  
For pulsars, **spin period** (`period`, in seconds) is central. Values span many orders of magnitude, so we plot **`log10(period)`** for positive periods and print `describe()` on the raw seconds.

In [ ]:
period_s = pulsars["period"].dropna()
period_s = period_s[period_s > 0]
log_period = np.log10(period_s)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_period, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(period / seconds)")
ax.set_ylabel("Number of pulsars")
ax.set_title("Distribution of spin period (ATNF pulsars)")
plt.show()

print("Numeric summary of period (seconds):")
print(pulsars["period"].describe())

**Filter to a meaningful subset. What's in it?**  
Many rows have no flux at every frequency. Here we keep only pulsars with a **measured 1400 MHz flux** (`flux_1400_mhz` not missing) and show a few useful columns.

In [ ]:
with_flux = pulsars[pulsars["flux_1400_mhz"].notna()].copy()
print(f"Subset size: {len(with_flux)} rows (from {len(pulsars)} total)")
print("Sample columns from the subset:")
with_flux[["name", "pulsar_type", "dm", "flux_1400_mhz", "discovery_survey"]].head(8)

**Group by a category and find the average of a numeric column.**  
We group by **`pulsar_type`** and take the mean of **`dm`** (dispersion measure, pc/cm³), a standard radio-pulsar observable.

In [ ]:
by_type = (
    pulsars.groupby("pulsar_type", dropna=False)["dm"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print("Mean DM (pc/cm^3) by pulsar_type:")
print(by_type)

**Where are the missing values? Are any columns incomplete?**  
We count `NaN` per column and show how many columns have gaps, plus the worst-off columns.

In [ ]:
missing_per_col = pulsars.isna().sum()
cols_with_gaps = (missing_per_col > 0).sum()
print(
    f"Columns with at least one missing value: {cols_with_gaps} "
    f"(out of {pulsars.shape[1]} total columns)"
)
print("\nTop 15 columns by number of missing cells:")
print(missing_per_col.sort_values(ascending=False).head(15))